In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import optuna
import lightgbm as lgb
import shap

# Download latest version
path = kagglehub.dataset_download("mohamedesam10/used-cars-dataset")

print("Path to dataset files:", path)
df = pd.read_csv('/kaggle/input/datasets/mohamedesam10/used-cars-dataset/used_cars.csv')
print(df.head())
print(df.info())
print(df.describe())
print(df.isnull().sum())
print(df.isnull().sum() / len(df) * 100)
df = df.drop(columns=['id'])

df['fuel_type'] = df['fuel_type'].fillna('Unknown')
print(df['fuel_type'].isnull().sum())

df['accident'] = df['accident'].fillna('None reported')
print(df['accident'].isnull().sum())

df['clean_title'] = df['clean_title'].fillna('No')
print(df['clean_title'].isnull().sum())

print(df['engine'])

df['horsepower'] = df['engine'].str.extract(r'(\d+\.?\d*)HP').astype(float)
df['engine_volume'] = df['engine'].str.extract(r'(\d+\.?\d*)L').astype(float)

df = df.drop(columns=['engine'])

print(df[['horsepower', 'engine_volume']].head())
print(df[['horsepower', 'engine_volume']].isnull().sum())

df['horsepower'] = df['horsepower'].fillna(df['horsepower'].median())
df['engine_volume'] = df['engine_volume'].fillna(df['engine_volume'].median())
print(df[['horsepower', 'engine_volume']].isnull().sum())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(df['price'], bins=50)
ax2.hist(np.log1p(df['price']), bins=50)
ax1.set_title('Распределение price')
ax2.set_title('Распределение log(price)')
plt.tight_layout()
plt.savefig('reports/figures/price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.boxplot(df['price'], vert=False)
ax.set_title('Анализ выбросов')
plt.tight_layout()
plt.savefig('reports/figures/price_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df['milage'], df['price'], alpha=0.1, s=1)
ax.set_xlabel('Пробег')
ax.set_ylabel('Цена')
ax.set_title('Зависимость цены от пробега')
plt.tight_layout()
plt.savefig('reports/figures/milage_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()

num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(num_cols)
print(cat_cols)

plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Корреляция числовых признаков')
plt.tight_layout()
plt.savefig('reports/figures/correlation.png', dpi=150, bbox_inches='tight')
plt.show()

brand_price = df.groupby('brand')['price'].mean().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(brand_price.index[::-1], brand_price.values[::-1], color='steelblue')
ax.set_xlabel('Средняя цена ($)')
ax.set_title('Топ-10 брендов по средней цене')
plt.tight_layout()
plt.savefig('reports/figures/top_brands.png', dpi=150, bbox_inches='tight')
plt.show()

X = df.drop(columns=['price'])
y = df['price']
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

threshold = y_train.quantile(0.99)
train_mask = y_train <= threshold
X_train = X_train[train_mask]
y_train = y_train[train_mask]

y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)
y_test_log = np.log1p(y_test)

print(f'train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}')
print(f'порог (99% train): {threshold:.0f}')
print(f'log_price train: min={y_train_log.min():.2f}, max={y_train_log.max():.2f}, mean={y_train_log.mean():.2f}')

cat_cols = X_train.select_dtypes(include='object').columns.tolist()
print(f'Категориальных: {len(cat_cols)}')
print(cat_cols)

for col in cat_cols:
    X_train[col] = X_train[col].astype('category')
    X_val[col] = X_val[col].astype('category')
    X_test[col] = X_test[col].astype('category')

from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train_enc = encoder.fit_transform(X_train[cat_cols])
X_val_enc = encoder.transform(X_val[cat_cols])
X_test_enc = encoder.transform(X_test[cat_cols])
X_train_concat = pd.concat([X_train.drop(columns=cat_cols).reset_index(drop=True), pd.DataFrame(X_train_enc, columns=cat_cols)], axis=1)
X_val_concat = pd.concat([X_val.drop(columns=cat_cols).reset_index(drop=True), pd.DataFrame(X_val_enc, columns=cat_cols)], axis=1)
X_test_concat = pd.concat([X_test.drop(columns=cat_cols).reset_index(drop=True), pd.DataFrame(X_test_enc, columns=cat_cols)], axis=1)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_concat)
X_val_scaled = scaler.transform(X_val_concat)
X_test_scaled = scaler.transform(X_test_concat)

model_dict = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'LGBM': LGBMRegressor(random_state=42, verbose=-1),
    'XGB': XGBRegressor(random_state=42, enable_categorical=True),
    'CatBoost': CatBoostRegressor(random_state=42, verbose=0, cat_features=cat_cols)
}


def model_train(model_name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    predict_log = model.predict(X_test)
    r2 = r2_score(y_test, predict_log)
    y_test_usd = np.expm1(y_test)
    predict_usd = np.expm1(predict_log)
    mae = mean_absolute_error(y_test_usd, predict_usd)
    rmse = np.sqrt(mean_squared_error(y_test_usd, predict_usd))
    baseline_results[model_name] = {
        'R2': r2,
        'MAE': mae,
        'RMSE': rmse
    }


baseline_results = {}
for model_name, model in model_dict.items():
    if model_name == 'Linear Regression':
        model_train(model_name, model, X_train_scaled, X_test_scaled, y_train_log, y_test_log)
    elif model_name == 'Random Forest':
        model_train(model_name, model, X_train_concat, X_test_concat, y_train_log, y_test_log)
    else:
        model_train(model_name, model, X_train, X_test, y_train_log, y_test_log)

pd.DataFrame(baseline_results).T.to_csv('reports/baseline_test_results.csv')


def objective(trial, model_type):
    if model_type == 'LGBM':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 2000, step=100),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'num_leaves': trial.suggest_int('num_leaves', 15, 127),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        }
        model = LGBMRegressor(random_state=42, verbose=-1, **params)
        model.fit(
            X_train, y_train_log,
            eval_set=[(X_val, y_val_log)],
            eval_metric='rmse',
            callbacks=[lgb.early_stopping(50, verbose=False)]
        )
        pred = model.predict(X_val)

    elif model_type == 'XGB':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 2000, step=100),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        }
        model = XGBRegressor(random_state=42, enable_categorical=True, **params)
        model.fit(X_train, y_train_log)
        pred = model.predict(X_val)

    elif model_type == 'CatBoost':
        params = {
            'iterations': trial.suggest_int('iterations', 200, 2000, step=100),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'depth': trial.suggest_int('depth', 3, 12),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
        }
        model = CatBoostRegressor(random_state=42, verbose=0, cat_features=cat_cols, **params)
        model.fit(X_train, y_train_log)
        pred = model.predict(X_val)

    elif model_type == 'Random Forest':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
            'max_depth': trial.suggest_int('max_depth', 3, 20),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        }
        model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
        model.fit(X_train_concat, y_train_log)
        pred = model.predict(X_val_concat)

    else:
        alpha = trial.suggest_float('alpha', 1e-4, 10.0, log=True)
        model = Ridge(alpha=alpha, random_state=42)
        model.fit(X_train_scaled, y_train_log)
        pred = model.predict(X_val_scaled)

    return r2_score(y_val_log, pred)


optuna.logging.set_verbosity(optuna.logging.WARNING)

studies = {}
for model_name in model_dict.keys():
    print(f'Обучение {model_name}')
    study = optuna.create_study(direction='maximize')
    study.optimize(
        lambda trial: objective(trial, model_name),
        n_trials=30,
        show_progress_bar=True
    )
    studies[model_name] = study
    print(f'{model_name}: best R2 = {study.best_value:.4f}\n')

for col in cat_cols:
    X_test[col] = pd.Categorical(X_test[col], categories=X_train[col].cat.categories)
    X_val[col] = pd.Categorical(X_val[col], categories=X_train[col].cat.categories)
for col in cat_cols:
    fill_value = X_train[col].mode()[0]
    X_val[col] = X_val[col].fillna(fill_value)
    X_test[col] = X_test[col].fillna(fill_value)
results = []
y_test_usd = np.expm1(y_test_log)

for model_name, study in studies.items():
    best_params = study.best_params

    if model_name == 'LGBM':
        model = LGBMRegressor(random_state=42, verbose=-1, **best_params)
        model.fit(X_train, y_train_log)
        pred_log = model.predict(X_test)
    elif model_name == 'XGB':
        model = XGBRegressor(random_state=42, enable_categorical=True, **best_params)
        model.fit(X_train, y_train_log)
        pred_log = model.predict(X_test)
    elif model_name == 'CatBoost':
        model = CatBoostRegressor(random_state=42, verbose=0, cat_features=cat_cols, **best_params)
        model.fit(X_train, y_train_log)
        pred_log = model.predict(X_test)
    elif model_name == 'Random Forest':
        model = RandomForestRegressor(random_state=42, n_jobs=-1, **best_params)
        model.fit(X_train_concat, y_train_log)
        pred_log = model.predict(X_test_concat)
    else:
        model = Ridge(random_state=42, **best_params)
        model.fit(X_train_scaled, y_train_log)
        pred_log = model.predict(X_test_scaled)

    r2 = r2_score(y_test_log, pred_log)

    pred_usd = np.expm1(pred_log)
    mae = mean_absolute_error(y_test_usd, pred_usd)
    rmse = np.sqrt(mean_squared_error(y_test_usd, pred_usd))

    results.append({
        'model': model_name,
        'R2': r2,
        'MAE': mae,
        'RMSE': rmse
    })

optuna_results = pd.DataFrame(results)
optuna_results.to_csv('reports/optuna_test_results.csv', index=False)

baseline = pd.read_csv('reports/baseline_test_results.csv', index_col=0)
optuna = pd.read_csv('reports/optuna_test_results.csv', index_col=0)

final_table = pd.DataFrame({
    'model': baseline.index,
    'baseline': baseline['R2'].values,
    'optuna': optuna['R2'].values
})

final_table['delta'] = final_table['optuna'] - final_table['baseline']
final_table.to_csv('reports/final_table.csv', index=False)
print(final_table.round(5))

best_model = XGBRegressor(random_state=42, enable_categorical=True, **studies['XGB'].best_params)
best_model.fit(X_train, y_train_log)
explainer = shap.TreeExplainer(best_model)
X_test_sample = X_test.sample(5000, random_state=42)
shap_values = explainer(X_test_sample)

shap.summary_plot(shap_values, X_test_sample, show=False, max_display=15)
plt.tight_layout()
plt.savefig('reports/figures/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

shap.waterfall_plot(shap_values[0], show=False, max_display=15)
plt.tight_layout()
plt.savefig('reports/figures/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
